# Pipeline project yeast


## 1. Trimming_pysam + indexing bam file
Trimming the last 30 nucleotides from aligned long reads in Scer.sorted.bam, considering only reads with FLAG values 0 (forward strand) and 16 (reverse strand), using pysam.

In [ ]:
!pip install biopython
!pip install pysam


import pysam
from Bio.Seq import Seq  # for reverse-complement

input_bam = "/home/saif/Documents/Saif_project_Yeast/Inputs/Saif/longReads/Scer.sorted.bam"
output_bam = "/home/ldufour/Documents/Saif_project_Yeast/Trimming/output_reverseCorrected_last30.bam"
read_length = 30

with pysam.AlignmentFile(input_bam, "rb") as infile, \
     pysam.AlignmentFile(output_bam, "wb", template=infile) as outfile:

    for read in infile.fetch():
        if read.flag not in {0, 16}:
            continue  # keep only forward (0) and reverse (16)

        if read.query_length < read_length:
            continue

        # Get the aligned pairs
        aligned_pairs = read.get_aligned_pairs(matches_only=True)
        if len(aligned_pairs) < read_length:
            continue

        # Prepare new read object
        new_read = read.__copy__()

        if read.flag == 0:
            # Forward strand: last 30 nt from the 3' end
            trimmed_seq = read.query_sequence[-read_length:]
            trimmed_qual = read.query_qualities[-read_length:] if read.query_qualities else None
            new_start = aligned_pairs[-read_length][1]
            new_read.pos = new_start
        else:
            # Reverse strand: last 30 nt is from the 5' end (in read space), reverse complement it
            trimmed_seq = str(Seq(read.query_sequence[:read_length]).reverse_complement())
            if read.query_qualities:
                trimmed_qual = read.query_qualities[:read_length][::-1]
            else:
                trimmed_qual = None
            new_start = aligned_pairs[read_length - 1][1]  # start at the last of the first 30 aligned bases
            new_read.pos = new_start

        # Update the new read
        new_read.query_sequence = trimmed_seq
        new_read.query_qualities = trimmed_qual
        new_read.cigar = [(0, read_length)]  # 30M match
        outfile.write(new_read)

print("Output saved to:", output_bam)


In [ ]:
%%bash 

module load SAMtools/1.19.2-GCC-13.3.0

samtools sort -o  /home/ldufour/Documents/Saif_project_Yeast/Trimming/output_reverseCorrected_last30_sorted.bam \
/home/ldufour/Documents/Saif_project_Yeast/Trimming/output_reverseCorrected_last30.bam

samtools index /home/ldufour/Documents/Saif_project_Yeast/Trimming/output_reverseCorrected_last30_sorted.bam

## 2. Peak Caller: MASC2

In [ ]:
%%bash
## peak controller. MASC2
source /home/saif/anaconda3/bin/activate MASC2 && macs2 callpeak \ 
-t /home/ldufour/Documents/Saif_project_Yeast/Trimming/output_reverseCorrected_last30_sorted.bam \
--outdir /home/ldufour/Documents/Saif_project_Yeast/Output/MASC2/ \
-n last30_2 \
--nomodel \
--extsize 30 \
--keep-dup auto \
-B --SPMR

## 3. Creating an annotation file for new transcripts

In [ ]:
!pip install gffutils

import pandas as pd
import gffutils
import os

# === File paths ===
GFF_FILE = "/home/saif/Documents/Saif_project_Yeast/Inputs/Saif/Annotation/Scer.utr.agat.gff"
NARROWPEAK_FILE = "/home/ldufour/Documents/Saif_project_Yeast/Output/MASC2/last30_2_peaks.narrowPeak"
DB_FILE = "/home/ldufour/Documents/Saif_project_Yeast/Output/New_annotation/narrowpeak_transcripts_trim30_2.db"
OUTPUT_GTF = "/home/ldufour/Documents/Saif_project_Yeast/Output/New_annotation/narrowpeak_transcripts_trim30_2.gtf"

# === Load .narrowPeak and compute summit position ===
narrow_df = pd.read_csv(NARROWPEAK_FILE, sep='\t', header=None,
                        names=["chrom", "start", "end", "name", "score", "strand",
                               "signalValue", "pValue", "qValue", "peak"])

# Calculate summit position
narrow_df["summit_pos"] = narrow_df["start"] + narrow_df["peak"]
print(f"Total peaks loaded: {len(narrow_df)}")

# === Build or load GFF database ===
if not os.path.exists(DB_FILE):
    print("Creating GFF database...")
    gffutils.create_db(GFF_FILE, DB_FILE, merge_strategy="merge", keep_order=True)
db = gffutils.FeatureDB(DB_FILE)

# === Match summit positions to transcripts and build new entries ===
new_gtf_lines = []
seen = set()

for _, row in narrow_df.iterrows():
    chrom = row["chrom"]
    summit = int(row["summit_pos"])
    peak_name = row["name"]

    # Only get features where summit lies within transcript
    overlapping = db.region(region=(chrom, summit, summit + 1), completely_within=False)

    for feature in overlapping:
        if feature.featuretype not in ["mRNA", "transcript"]:
            continue

        if not (feature.start <= summit <= feature.end):
            continue

        gene_id = feature.attributes.get("gene_id", [None])[0] or feature.id
        strand = feature.strand

        if strand == "+":
            tx_start = feature.start
            tx_end = summit
        elif strand == "-":
            tx_start = summit
            tx_end = feature.end
        else:
            continue

        new_tx_id = f"{gene_id}_{peak_name}"
        if new_tx_id in seen:
            continue
        seen.add(new_tx_id)

        line = (
            f"{chrom}\tcustom\ttranscript\t{tx_start}\t{tx_end}\t.\t{strand}\t.\t"
            f'gene_id "{gene_id}"; transcript_id "{new_tx_id}";\n'
        )
        new_gtf_lines.append(line)

# === Write updated GTF ===
with open(OUTPUT_GTF, "w") as f:
    f.writelines(new_gtf_lines)

print(f"Updated transcript annotations saved to {OUTPUT_GTF}")

### 7.6.7 To add median_polya_length and num_reads_with_polya from polya_stats_all.tsv to summary1_with_correct_read_counts.csv, you can perform a merge on gene_id and transcript_id.